In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
import attacks
from privacy_accountant import PrivacyAccountant
from tqdm import tqdm
from torchvision import datasets, transforms

In [3]:
use_cuda = False
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.MNIST('./mnist_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.MNIST('./mnist_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
attack = attacks.PGD()


In [4]:
def clip_gradients(gradient, gradient_norm_bound=1):
    return gradient * min(1, gradient_norm_bound/torch.norm(gradient, 2))

def gaussian_noise(gradient, noise, gradient_norm_bound = 1):
    gaussian_noise = torch.randn_like(gradient) * noise * gradient_norm_bound
    return gradient + gaussian_noise

In [7]:
def train_model(model, train_loader, num_epochs, noise, accountant: PrivacyAccountant, enable_defense=True, attack_type='pgd', eps=0.1):
    # TODO: implement this function that trains a given model on the MNIST dataset.
    # this is a general-purpose function for both standard training and adversarial training.
    # (toggle enable_defense parameter to switch between training schemes)
    model.train()
    # epsilons_clean = []
    lr = 1e-2
    losses = []
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    for epoch in tqdm(range(num_epochs)):
        for index, (images, labels) in enumerate(train_loader):
            logits = model(images)
            optimizer.zero_grad()

            loss = F.cross_entropy(logits, labels)
            loss.backward()

            for param in model.parameters():
                if param.grad is not None:
                    param.grad = clip_gradients(param.grad)
                    param.grad = gaussian_noise(param.grad, noise)

                    param.grad = param.grad / len(images)

            optimizer.step()
            accountant.step()
            losses.append(loss.item())
            

            if epoch % 10 == 0 and index == 0:
                epsilon = accountant.get_privacy_budget(target_delta=1e-5)
                print(f'For Epoch {epoch}, Batch {index} (Clean training) | e:{epsilon:.2f}, delta: {1e-5}')
                

        if enable_defense:
        # for epoch in range(num_epochs):
            for index, (images, labels) in enumerate(train_loader):
                adversary_images = attack.pgd_untargeted(model, images, labels, 10,  eps, 0.01)
                model.train()
                optimizer.zero_grad()
                logits = model(adversary_images)
                loss = F.cross_entropy(logits, labels)
                loss.backward()

                for param in model.parameters():
                    if param.grad is not None:
                        param.grad = clip_gradients(param.grad)
                        param.grad = gaussian_noise(param.grad, noise)
                        param.grad = param.grad / len(images)
                optimizer.step()
                accountant.step()

                losses.append(loss.item())
            if epoch % 10 == 0 and index == 0:
                epsilon = accountant.get_privacy_budget(target_delta=1e-5)
                print(f'For Epoch {epoch}, Batch {index} (Adv training) | e:{epsilon:.2f}, delta: {1e-5}')
                # print(f'Epoch [{epoch}/{num_epochs}] Loss = {loss.item():.3f}')

In [8]:
from model import fcNet
noise = 1.1
num_epochs = 100
q = train_loader.batch_size/len(train_loader.dataset)
accountant = PrivacyAccountant(noise, q)
fc_model = fcNet()

train_model(fc_model, train_loader, num_epochs, noise, accountant, enable_defense=True)

  0%|          | 0/100 [00:00<?, ?it/s]

For Epoch 0, Batch 0 (Clean training) | e:0.12, delta: 1e-05


 10%|█         | 10/100 [04:36<47:30, 31.67s/it]

For Epoch 10, Batch 0 (Clean training) | e:0.46, delta: 1e-05


 20%|██        | 20/100 [11:26<50:45, 38.07s/it]  

...

 80%|████████  | 80/100 [1:00:36<15:57, 47.87s/it]

For Epoch 80, Batch 0 (Clean training) | e:1.31, delta: 1e-05


 90%|█████████ | 90/100 [1:06:34<05:22, 32.27s/it]

For Epoch 90, Batch 0 (Clean training) | e:1.39, delta: 1e-05


100%|██████████| 100/100 [1:11:27<00:00, 42.88s/it]


In [9]:
torch.save(fc_model.state_dict(), f'dp-adv.pt')

In [10]:
correct = 0
fc_model.eval()
for j, (images, labels) in enumerate(test_loader):
  images, labels = images, labels
  logits = fc_model(images)
  _, preds = torch.max(logits, 1)
  correct += (preds == labels).sum().item()
  # print('Batch [{}/{}]'.format(j+1, len(test_loader)))
fc_model.train()
print('Accuracy = {}%'.format(float(correct) * 100 / 10000))

Accuracy = 18.83%


In [11]:
correct = 0
eps = 0.1
fc_model.eval()
for j, (images, labels) in enumerate(test_loader):
  images, labels = images, labels
  adv_images = attack.pgd_untargeted(fc_model, images, labels, 20, eps, 0.01)  
  logits = fc_model(images)
  adv_logits = fc_model(adv_images)
  _, preds = torch.max(logits, 1)
  _, adv_preds = torch.max(adv_logits, 1)
  correct += (preds == labels).sum().item()
  correct += (adv_preds == labels).sum().item()
  # print('Batch [{}/{}]'.format(j+1, len(test_loader)))
fc_model.train()
print('Accuracy = {}%'.format(float(correct) * 100 / 20000))

Accuracy = 14.515%


In [1]:
fc_model

NameError: name 'fc_model' is not defined